# Benchmarking a single RTX 8000

Benchmark in PyTorch is straightforward, and now that your system is running cleanly with one GPU, you can get a precise performance profile with a few targeted tests. Below is a ready‑to‑run benchmark suite that covers the three things that actually matter:

- raw tensor throughput
- matrix‑multiply (GEMM) performance
- end‑to‑end transformer workload speed

This gives you a realistic picture of how your GPU performs in your actual ANIMA/LLM workflows.

In [1]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_capability(0))


True
Quadro RTX 8000
(7, 5)


### 🧪 Benchmark 1 — Raw tensor throughput (memory + compute)
This measures how fast the GPU can push large tensors around.

In [2]:
import torch, time

device = torch.device("cuda:0")

def bench_tensor_ops(size=4096):
    a = torch.randn(size, size, device=device)
    b = torch.randn(size, size, device=device)

    torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(50):
        c = a + b
        d = a * b
    torch.cuda.synchronize()
    return time.time() - t0

print("Tensor ops time:", bench_tensor_ops())

Tensor ops time: 0.09608125686645508


In [3]:
# This gives you a baseline for memory bandwidth and elementwise ops.

import torch, time

device = torch.device("cuda:0")

def bench_tensor_ops(size=4096):
    a = torch.randn(size, size, device=device)
    b = torch.randn(size, size, device=device)

    torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(50):
        c = a + b
        d = a * b
    torch.cuda.synchronize()
    return time.time() - t0

print("Tensor ops time:", bench_tensor_ops())

Tensor ops time: 0.2551867961883545


## 🧮 Benchmark 2 — GEMM (matrix multiply) throughput
This is the real heart of transformer workloads.

In [4]:
# This is the closest proxy to attention + MLP layers.

import torch, time

device = torch.device("cuda:0")

def bench_gemm(size=4096):
    a = torch.randn(size, size, device=device)
    b = torch.randn(size, size, device=device)

    torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(20):
        c = torch.matmul(a, b)
    torch.cuda.synchronize()
    return time.time() - t0

print("GEMM time:", bench_gemm())

GEMM time: 0.36530137062072754


## 🤖 Benchmark 3 — Mini‑Transformer forward pass
This is the most realistic test for your ANIMA workflows.

This simulates:

- batch size 32
- sequence length 512
- hidden size 1024
- 16‑head attention

A realistic mid‑size transformer layer.

## 📊 What performance numbers to expect
For a single RTX 8000:

- Tensor ops: fast, usually < 0.2s
- GEMM: 0.3–0.6s depending on size
- Mini‑Transformer: 0.4–0.8s for the above config

If your numbers fall in these ranges, your GPU is performing perfectly.

In [6]:
import torch, time
import torch.nn as nn

device = torch.device("cuda:0")

class MiniTransformer(nn.Module):
    def __init__(self, d=1024, h=16):
        super().__init__()
        self.attn = nn.MultiheadAttention(d, h, batch_first=True)
        self.ff = nn.Sequential(
            nn.Linear(d, 4*d),
            nn.ReLU(),
            nn.Linear(4*d, d)
        )

    def forward(self, x):
        y, _ = self.attn(x, x, x)
        return self.ff(y)

model = MiniTransformer().to(device)
x = torch.randn(32, 512, 1024, device=device)

torch.cuda.synchronize()
t0 = time.time()
for _ in range(20):
    y = model(x)
torch.cuda.synchronize()

print("Transformer forward time:", time.time() - t0)

Transformer forward time: 1.086355209350586
